# Workspace configuration checks

Offline regression checks for relocation, null/missing assets, invalid settings and duplicate YAML keys. Uses temporary fixtures only; no simulator or runtime dependency.


In [1]:
from pathlib import Path
import os
_candidate = Path(os.environ.get('TRACE_LAB_ROOT', Path.cwd())).expanduser().resolve()
_candidates = [_candidate] if os.environ.get('TRACE_LAB_ROOT') else [_candidate, *_candidate.parents]
ROOT = next((p for p in _candidates if (p / '.trace-lab-root').is_file() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open inside trace-lab or set TRACE_LAB_ROOT to the clone')
# Run definition cells in this fresh kernel; this notebook has no execution side effects.
get_ipython().run_line_magic('run', '"' + str(ROOT / 'notebooks/library/configuration.ipynb') + '"')
ROOT = workspace_root(ROOT)

print('Workspace ready:', ROOT)


Configuration helpers loaded. No runtime probe or installation has been executed.
Workspace ready: /home/obidit/900_PrePreDefense/trace-lab


In [2]:
import tempfile
import unittest

class WorkspaceConfigurationTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.root = Path(self.temp.name)
        (self.root / 'configs').mkdir()
        (self.root / '.trace-lab-root').write_text('test')
        self.data = {'schema_version': 1, **{key: None for key in ASSET_KINDS}}
    def tearDown(self):
        self.temp.cleanup()
    def config(self):
        path = self.root / 'configs/assets.example.yaml'
        path.write_text(yaml.safe_dump(self.data))
        return path
    def test_relative_paths_survive_relocation(self):
        self.data['model_directory'] = 'assets/models'
        loaded = load_assets(self.root, self.config())
        self.assertEqual(loaded['paths']['model_directory'], self.root / 'assets/models')
    def test_unconfigured_is_distinct_from_missing(self):
        self.data['model_directory'] = 'missing'
        states = {row['asset']: row['status'] for row in asset_status(load_assets(self.root, self.config()))}
        self.assertEqual(states['runtime_python'], 'UNCONFIGURED')
        self.assertEqual(states['model_directory'], 'MISSING')
    def test_directory_cannot_be_runtime_executable(self):
        self.data['runtime_python'] = str(self.root)
        assets = load_assets(self.root, self.config())
        with self.assertRaises(ConfigurationError): require_asset(assets, 'runtime_python')
    def test_unknown_keys_rejected(self):
        self.data['runtime_pythno'] = 'wrong'
        with self.assertRaises(ConfigurationError): load_assets(self.root, self.config())
    def test_boolean_schema_rejected(self):
        self.data['schema_version'] = True
        with self.assertRaises(ConfigurationError): load_assets(self.root, self.config())
    def test_duplicate_yaml_rejected(self):
        path = self.config(); path.write_text('schema_version: 1\nschema_version: 2\n')
        with self.assertRaises(ConfigurationError): read_yaml(path)
    def test_unresolved_variable_rejected(self):
        with self.assertRaises(ConfigurationError): resolve_path('${TRACE_LAB_INTENTIONALLY_MISSING}/python', self.root)
    def test_explicit_missing_config_does_not_fallback(self):
        self.config()
        with self.assertRaises(FileNotFoundError): load_assets(self.root, self.root / 'absent.yaml')
    def test_runtime_mismatch_is_not_pass(self):
        example = {'packages': {}, 'python_version': '3.13.0', 'pip_check_exit_code': 0,
                   'cpu_loss': 14., 'cpu_gradient': [2.,4.,6.], 'device': 'cpu', 'torch_version':'wrong'}
        self.assertEqual(compare_runtime(ROOT, example)['status'], 'FAIL')

suite = unittest.defaultTestLoader.loadTestsFromTestCase(WorkspaceConfigurationTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Configuration regression failure'

print(f'PASS: {result.testsRun} configuration regression tests; failures={len(result.failures)}, errors={len(result.errors)}')


test_boolean_schema_rejected (__main__.WorkspaceConfigurationTests.test_boolean_schema_rejected) ... 

ok


test_directory_cannot_be_runtime_executable (__main__.WorkspaceConfigurationTests.test_directory_cannot_be_runtime_executable) ... 

ok


test_duplicate_yaml_rejected (__main__.WorkspaceConfigurationTests.test_duplicate_yaml_rejected) ... 

...

ok


test_unresolved_variable_rejected (__main__.WorkspaceConfigurationTests.test_unresolved_variable_rejected) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.009s

OK


PASS: 9 configuration regression tests; failures=0, errors=0
